In [1]:
import math
import pandas as pd

#        -        Node Class        -
class Node:
    def __init__(self, attribute=None, label=None):
        self.attribute = attribute
        self.label = label
        self.children = {}

    @property
    def is_leaf(self):
        return self.label is not None


#       -         DecisionTree Class      -
class DecisionTree:
    def __init__(self, target, criterion="ig", max_depth=None):
        """
        target: column name of labels - True/False or M/B
        criterion: 'ig' (information gain) or 'gini'
        max_depth: maximum depth of tree (None = unlimited)
        """
        self.target = target
        self.criterion = criterion
        self.max_depth = max_depth
        self.tree = None

    def fit(self, df, attributes):
        majority = self.plurality(df[self.target])
        self.tree = self.build_tree(df, attributes, majority, depth=0)

    def predict_one(self, row):
        node = self.tree
        while not node.is_leaf:
            val = row[node.attribute]
            if val in node.children:
                node = node.children[val]
            else:
                return node.label
        return node.label

    def predict(self, df):
        return df.apply(self.predict_one, axis=1)

    #        -       Tree building            -
    def build_tree(self, df, attributes, parent_label, depth):
        y = df[self.target]

        # Case 1: if dataset is empty
        if df.empty:
            return Node(label=parent_label)

        # Case 2: if all labels are same
        if y.nunique() == 1:
            return Node(label=y.iloc[0])

        # Case 3: if no attributes left OR max_depth reached
        if not attributes or (self.max_depth is not None and depth >= self.max_depth):
            return Node(label=self.plurality(y))

        # Choose best attribute or root node
        best_attr = self.choose_best(df, attributes)
        node = Node(attribute=best_attr)

        # Recurse tree building o
        for v in df[best_attr].unique():
            subset = df[df[best_attr] == v]
            child = self.build_tree(
                subset,
                [a for a in attributes if a != best_attr],
                self.plurality(y),
                depth + 1
            )
            node.children[v] = child

        return node

    #          -      Attribute selection      -
    def choose_best(self, df, attributes):
        if self.criterion == "ig":
            best_attr = None 
            best_gain = -float("inf")
            for attr in attributes:
                gain = self.info_gain(df, attr)
                if gain > best_gain:
                    best_attr = attr
                    best_gain =  gain
            return best_attr

        elif self.criterion == "gini":
            best_attr = None 
            best_gini = float("inf")
            for attr in attributes:
                gini_value = self.gini_index(df, attr)
                if gini_value < best_gini:
                    best_attr = attr
                    best_gini = gini_value
            return best_attr

        else:
            raise ValueError("Unknown criterion")

    #          -      Metrics Calculation         -
    def entropy(self, series):
        probs = series.value_counts(normalize=True)
        return -sum(p * math.log2(p) for p in probs if p > 0)

    def info_gain(self, df, attr):
        total_entropy = self.entropy(df[self.target])
        weighted = 0
        for v in df[attr].unique():
            subset = df[df[attr] == v]
            weighted += (len(subset)/len(df)) * self.entropy(subset[self.target])
        return total_entropy - weighted

    def gini(self, series):
        probs = series.value_counts(normalize=True)
        return 1 - sum(p**2 for p in probs)

    def gini_index(self, df, attr):
        weighted = 0
        for v in df[attr].unique():
            subset = df[df[attr] == v]
            weighted += (len(subset)/len(df)) * self.gini(subset[self.target])
        return weighted

    def plurality(self, series):
        return series.value_counts().idxmax()

    ##        -        Print tree       -
    def print_tree(self, node=None, depth=0):
        if node is None:
            node = self.tree

        indent = "  " * depth
        if node.is_leaf:
            print(f"{indent}Leaf → {node.label}")
        else:
            print(f"{indent}[{node.attribute}]")
            for val, child in node.children.items():
                print(f"{indent} ├─ {node.attribute} = {val}")
                self.print_tree(child, depth + 2)

#           -          Performance Metrics        -
def evaluate_metrics(df, preds, target, positive_label="M"):
    y_true = df[target]

    # Accuracy + Error
    acc = (preds == y_true).mean()
    err = 1 - acc

    # Precision - of all predicted positives, what proportion is correct?
    tp = sum((yt == positive_label) and (yp == positive_label)
             for yt, yp in zip(y_true, preds))
    fp = sum((yt != positive_label) and (yp == positive_label)
             for yt, yp in zip(y_true, preds))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Recall - of all actual positives, what proportion is predicted?
    fn = sum((yt == positive_label) and (yp != positive_label)
             for yt, yp in zip(y_true, preds))
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return acc, err, prec, rec


def build_results_table(train_df, test_df, target, attributes, positive_label="M"):
    results = []

    # IG Tree
    dt_ig = DecisionTree(target=target, criterion="ig")
    dt_ig.fit(train_df, attributes)
    preds_ig = dt_ig.predict(test_df)
    results.append(["DecisionTree-IG"] + list(evaluate_metrics(test_df, preds_ig, target, positive_label)))

    # Gini Tree
    dt_gini = DecisionTree(target=target, criterion="gini")
    dt_gini.fit(train_df, attributes)
    preds_gini = dt_gini.predict(test_df)
    results.append(["DecisionTree-Gini"] + list(evaluate_metrics(test_df, preds_gini, target, positive_label)))

    # Majority Baseline
    majority_class = train_df[target].value_counts().idxmax()
    preds_base = [majority_class] * len(test_df)
    results.append(["Majority Baseline"] + list(evaluate_metrics(test_df, preds_base, target, positive_label)))

    return pd.DataFrame(results, columns=["Model", "Accuracy", "Error", "Precision", "Recall"])


# ------------------- Problem 2 usage -------------------
def problem2_example(train_df, target):
    attributes = [c for c in train_df.columns if c != target]
    criterion = input("Enter criterion (ig for Information Gain, gini for Gini Index): ")
    dt = DecisionTree(target=target, criterion=criterion)
    dt.fit(train_df, attributes)
    dt.print_tree()
    return dt


# ------------------- Problem 3 (with dev set) -------------------
def problem3_example(train_df, dev_df, test_df, target, positive_label="M"):
    attributes = [c for c in train_df.columns if c != target]

    criteria = ["ig", "gini"]
    depths = [None, 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
    best_acc = 0
    best_params = None

    tuning_results = []

    # --------- Step 1: Train on train set and Tune on dev set ---------
    for crit in criteria:
        for depth in depths:
            dt = DecisionTree(target=target, criterion=crit, max_depth=depth)
            dt.fit(train_df, attributes)
            preds_dev = dt.predict(dev_df)

            acc_dev, err_dev, prec_dev, rec_dev = evaluate_metrics(
                dev_df, preds_dev, target, positive_label
            )
    
            print(f"Criterion={crit}, Depth={depth}, "
                  f"Acc={acc_dev:.3f}, Prec={prec_dev:.3f}, Rec={rec_dev:.3f}")
        
            tuning_results.append([crit, depth, acc_dev, err_dev, prec_dev, rec_dev])

            if acc_dev > best_acc:
                best_acc = acc_dev
                best_params = (crit, depth)

    tuning_df = pd.DataFrame(
        tuning_results,
        columns=["Criterion", "Depth", "Accuracy", "Error", "Precision", "Recall"]
    )

    print("\nBest hyperparameters (from dev):", best_params, "Dev Accuracy:", round(best_acc, 3))

    # --------- Step 2: Retrain with best params ---------
    best_crit = best_params[0]
    best_depth = best_params[1]
    dt_final = DecisionTree(target=target, criterion=best_crit, max_depth=best_depth)
    dt_final.fit(train_df, attributes)

    # --------- Step 3: Build final results table ---------
    results = []

    # Final tuned tree
    preds_final = dt_final.predict(test_df)
    results.append([f"DecisionTree-{best_crit.upper()} (Best)",
                    *evaluate_metrics(test_df, preds_final, target, positive_label)])

    # Gini result
    dt_gini = DecisionTree(target=target, criterion="gini")
    dt_gini.fit(train_df, attributes)
    preds_gini = dt_gini.predict(test_df)
    results.append(["DecisionTree-GINI", *evaluate_metrics(test_df, preds_gini, target, positive_label)])

    # IG result
    dt_ig = DecisionTree(target=target, criterion="ig")
    dt_ig.fit(train_df, attributes)
    preds_ig = dt_ig.predict(test_df)
    results.append(["DecisionTree-IG", *evaluate_metrics(test_df, preds_ig, target, positive_label)])

    # Majority baseline result
    majority_class = train_df[target].value_counts().idxmax()
    preds_base = [majority_class] * len(test_df)
    results.append(["Majority Baseline", *evaluate_metrics(test_df, preds_base, target, positive_label)])

    table = pd.DataFrame(results, columns=["Model", "Accuracy", "Error", "Precision", "Recall"])
    print("\nFinal Test Results:")
    print(table.round(3))

    return tuning_df, table

In [2]:
train_df = pd.read_csv("wdbc_train.csv")
dev_df   = pd.read_csv("wdbc_dev.csv")
test_df  = pd.read_csv("wdbc_test.csv")
target = "Diagnosis"
tuning_df, results_table = problem3_example(train_df, dev_df, test_df, target, positive_label="M")

# To see all dev set results:
print(tuning_df.head())

# To save tuning results to CSV for report:
tuning_df.to_csv("dev_tuning_results.csv", index=False)


Criterion=ig, Depth=None, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=1, Acc=0.947, Prec=0.894, Rec=0.977
Criterion=ig, Depth=2, Acc=0.982, Prec=0.956, Rec=1.000
Criterion=ig, Depth=3, Acc=0.965, Prec=0.976, Rec=0.953
Criterion=ig, Depth=4, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=5, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=6, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=7, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=8, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=9, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=10, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=11, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=12, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=13, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=14, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=ig, Depth=15, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=gini, Depth=None, Acc=0.947, Prec=0.975, Rec=0.907
Criterion=gini, Depth=1, Acc=0.947